# TRIAGE-EG Notebook 41 — Trial P1 R4 Surgical Final Repair

Diagnostic-only 24-query R4 run. It carries R3 KIS and Event Graph semantics unchanged, enforces coordinate-exact frozen BCF1 SAFE Top5 tuples, applies independent QA context/support/type gates, uses the four-field Qwen schema, and performs only query-local candidate/neighbor OCR rescue. OJ-ready ZIPs are emitted only after every hard gate passes. It never opens GT, runs Whisper, starts a corpus job, uploads a submission, or changes production policy.


In [ ]:
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git"
REPO_REF = "TRIAGEEG"
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
RAW_INPUT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
TRIAL_INPUT = Path(
    os.environ.get("AIC_TRIAL_ROOT", "/kaggle/input/datasets/irthn1311/thunghiem-bo-de-thi")
)
BCF1_INPUT = Path(
    os.environ.get(
        "AIC_TRIAL_BCF1_ROOT", "/kaggle/input/datasets/irthn1311/trial-p1-true-bcf1-bundle"
    )
)
ASR_INPUT = Path(
    os.environ.get(
        "AIC_ASR_EXTERNAL_V3_ROOT", "/kaggle/input/datasets/irthn1311/asr-external-v3-validated"
    )
)
EXTERNAL_INPUT = Path(
    os.environ.get(
        "AIC_EXTERNAL_RUNTIME_EVIDENCE_ROOT",
        "/kaggle/input/datasets/irthn1311/external-multimodal-runtime-evidence-v3",
    )
)
E5_INPUT = Path(
    os.environ.get(
        "AIC_E5_QUERY_ENCODER_ROOT",
        "/kaggle/input/datasets/irthn1311/aic2026-multilingual-e5-small-onnx-query-encoder",
    )
)
XCLIP_INPUT = Path(
    os.environ.get(
        "AIC_XCLIP_ROOT", "/kaggle/input/datasets/irthn1311/fs1-xclip-base-patch32-asset"
    )
)
QWEN_INPUT = Path(
    os.environ.get(
        "AIC_QWEN_ROOT", "/kaggle/input/datasets/irthn1311/fs1-qwen2-5-vl-3b-instruct-asset"
    )
)
OUTPUT_ROOT = Path("/kaggle/working/trial_p1_multimodal_r4")
OUTPUT_ZIP = Path("/kaggle/working/trial_p1_multimodal_r4_bundle.zip")
WORK_ROOT = Path("/kaggle/working/trial_p1_multimodal_r4_work")
for run_root in (OUTPUT_ROOT, WORK_ROOT):
    if run_root.exists():
        shutil.rmtree(run_root)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print(
    {
        "mode": "TRIAL_P1_R4_SURGICAL_FINAL_REPAIR",
        "required_inputs": {
            "raw_dataset": str(RAW_INPUT),
            "official_trial_package": str(TRIAL_INPUT),
            "frozen_true_bcf1": str(BCF1_INPUT),
            "asr_external_v3_validated": str(ASR_INPUT),
            "external_ocr_object_runtime_evidence": str(EXTERNAL_INPUT),
            "e5_query_encoder": str(E5_INPUT),
            "xclip_offline_asset": str(XCLIP_INPUT),
            "qwen_offline_asset": str(QWEN_INPUT),
        },
        "optional_inputs": {},
        "internet_required": "GIT_FETCH_AND_PINNED_ONNXRUNTIME_INSTALL_IF_MISSING",
        "model_download_required": False,
        "local_ocr": "BOUNDED_TESSERACT_ON_SHORTLISTED_CANDIDATE_NEIGHBORS_ONLY",
        "whisper_run": False,
        "corpus_job": False,
        "gpu": "ONE_T4_FOR_EXISTING_XCLIP_THEN_QWEN_SEQUENTIALLY",
        "gt_opened": False,
        "submission_uploaded": False,
        "output_zip": str(OUTPUT_ZIP),
    }
)

In [ ]:
import importlib
import subprocess
import sys
from importlib import metadata

ONNXRUNTIME_VERSION = "1.27.0"
try:
    installed_ort = metadata.version("onnxruntime")
except metadata.PackageNotFoundError:
    installed_ort = None
if installed_ort != ONNXRUNTIME_VERSION:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "--no-input",
            "--only-binary=:all:",
            f"onnxruntime=={ONNXRUNTIME_VERSION}",
        ],
        check=True,
    )
    importlib.invalidate_caches()
import onnxruntime as ort  # noqa: E402

if ort.__version__ != ONNXRUNTIME_VERSION:
    raise RuntimeError(f"ONNXRUNTIME_VERSION_MISMATCH:{ort.__version__}:{ONNXRUNTIME_VERSION}")
if "CPUExecutionProvider" not in ort.get_available_providers():
    raise RuntimeError(f"ONNXRUNTIME_CPU_PROVIDER_MISSING:{ort.get_available_providers()}")
print(
    {
        "onnxruntime": ort.__version__,
        "providers": ort.get_available_providers(),
        "tokenizers": metadata.version("tokenizers"),
        "dependency_install_performed": installed_ort != ONNXRUNTIME_VERSION,
        "model_download_performed": False,
    }
)

In [ ]:
import subprocess
import sys

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
GIT_STATUS = subprocess.check_output(["git", "status", "--short"], cwd=REPO_DIR, text=True).strip()
sys.path.insert(0, str(REPO_DIR / "src"))
import torch  # noqa: E402

if not torch.cuda.is_available():
    raise RuntimeError("TRIAL_MULTIMODAL_T4_REQUIRED")
print(
    {
        "source_ref": REPO_REF,
        "HEAD": HEAD,
        "checkout_mode": "DETACHED_FETCH_HEAD",
        "git_status": GIT_STATUS or "CLEAN",
        "gpu": torch.cuda.get_device_name(0),
    }
)

In [ ]:
import json
import re
import shutil
import zipfile


def bounded(root, name, max_depth=6):
    root = Path(root)
    found = []
    if not root.exists():
        return found
    for directory, subdirs, files in os.walk(root):
        current = Path(directory)
        depth = len(current.relative_to(root).parts)
        subdirs[:] = (
            []
            if depth >= max_depth
            else [item for item in subdirs if item not in {".cache", "blobs", "snapshots"}]
        )
        if name in files:
            found.append(current / name)
    return found


def normalized(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).casefold())


def mount_key(value):
    key = normalized(value)
    for suffix in ("bundle", "reports", "dataset"):
        if key.endswith(suffix):
            key = key[: -len(suffix)]
    return key


def roots_for(hint):
    if hint.exists():
        return [hint]
    target = mount_key(hint.name)
    roots = []
    search = Path("/kaggle/input")
    if search.exists():
        for directory, subdirs, _ in os.walk(search):
            current = Path(directory)
            depth = len(current.relative_to(search).parts)
            subdirs[:] = [] if depth >= 4 else subdirs
            if current.is_dir() and mount_key(current.name) == target:
                roots.append(current)
    return sorted(set(roots))


def one(root, name):
    root = Path(root)
    roots = roots_for(root)
    search_roots = roots or [Path("/kaggle/input")]
    search_depth = 6 if roots else 4
    values = sorted(
        set(path.resolve() for base in search_roots for path in bounded(base, name, search_depth))
    )
    if not values:
        zip_hits = []
        for base in search_roots:
            zip_paths = []
            for directory, subdirs, files in os.walk(base):
                current = Path(directory)
                depth = len(current.relative_to(base).parts)
                subdirs[:] = [] if depth >= search_depth else subdirs
                zip_paths.extend(
                    current / file for file in files if file.casefold().endswith(".zip")
                )
            for archive_path in zip_paths:
                with zipfile.ZipFile(archive_path) as archive:
                    members = [
                        member
                        for member in archive.namelist()
                        if Path(member).name == name
                        and not Path(member).is_absolute()
                        and ".." not in Path(member).parts
                        and "\\" not in member
                    ]
                    for member in members:
                        zip_hits.append((archive_path, member))
        if len(zip_hits) == 1:
            archive_path, member = zip_hits[0]
            extract_root = WORK_ROOT / "resolved" / normalized(archive_path.stem)
            extract_root.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(archive_path) as archive:
                names = archive.namelist()
                if len(names) != len(set(names)):
                    raise RuntimeError(f"Duplicate ZIP members: {archive_path}")
                for info in archive.infolist():
                    relative = Path(info.filename)
                    if relative.is_absolute() or ".." in relative.parts or "\\" in info.filename:
                        raise RuntimeError(f"Unsafe ZIP member: {info.filename}")
                    target = extract_root / relative
                    if info.is_dir():
                        target.mkdir(parents=True, exist_ok=True)
                        continue
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(info) as source, target.open("wb") as output:
                        shutil.copyfileobj(source, output)
            values = sorted(set(path.resolve() for path in bounded(extract_root, name)))
    if len(values) != 1:
        raise RuntimeError(
            f"Expected exactly one {name}; requested={root}; "
            f"searched={search_roots}; found={values}"
        )
    return values[0]


from triage_eg.trial_p1.multimodal_dryrun import write_blocked_artifacts  # noqa: E402

required = {
    "trial_mount": TRIAL_INPUT,
    "query_plans": (BCF1_INPUT, "trial_p1_query_plans_v2.jsonl"),
    "bcf1_predictions": (BCF1_INPUT, "trial_p1_BCF1_F1_predictions.jsonl"),
    "asr_provenance": (ASR_INPUT, "asr_external_v3_provenance.json"),
    "ocr_corpus": (EXTERNAL_INPUT, "ocr_records_external_v3.parquet"),
    "object_corpus": (EXTERNAL_INPUT, "object_records_external_v3.parquet"),
    "e5_model": (E5_INPUT, "model.onnx"),
    "xclip_config": (XCLIP_INPUT, "config.json"),
    "qwen_config": (QWEN_INPUT, "config.json"),
}
BLOCKERS = []
RESOLVED = {}
for label, value in required.items():
    try:
        if label == "trial_mount":
            mounts = roots_for(value)
            if len(mounts) != 1:
                raise RuntimeError(f"Expected one Trial mount; found {mounts}")
            RESOLVED[label] = mounts[0].resolve()
        else:
            RESOLVED[label] = one(value[0], value[1])
    except Exception as error:
        BLOCKERS.append(f"{label}: {type(error).__name__}: {error}")
if BLOCKERS:
    write_blocked_artifacts(
        OUTPUT_ROOT, BLOCKERS, {"HEAD": HEAD, "gt_opened": False, "submission_uploaded": False}
    )
    shutil.make_archive(str(OUTPUT_ZIP.with_suffix("")), "zip", OUTPUT_ROOT)
    raise RuntimeError(
        {"TRIAL_MULTIMODAL_PREFLIGHT_BLOCKED": BLOCKERS, "download_zip": str(OUTPUT_ZIP)}
    )
trial_zips = sorted(RESOLVED["trial_mount"].rglob("THUNGHIEM-bo-de-thi.zip"))
if len(trial_zips) == 1:
    TRIAL_ZIP = trial_zips[0]
    TRIAL_SOURCE_MODE = "ORIGINAL_ZIP"
elif not trial_zips:
    trial_txts = sorted(RESOLVED["trial_mount"].rglob("query-p1-*-*.txt"))
    trial_parents = {path.parent.resolve() for path in trial_txts}
    if len(trial_txts) != 24 or len(trial_parents) != 1:
        raise RuntimeError(
            f"Expanded Trial package contract failed: files={len(trial_txts)}, "
            f"parents={trial_parents}"
        )
    TRIAL_ZIP = WORK_ROOT / "official_trial" / "THUNGHIEM-bo-de-thi.zip"
    TRIAL_ZIP.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(TRIAL_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in trial_txts:
            archive.write(path, path.name)
    TRIAL_SOURCE_MODE = "KAGGLE_EXPANDED_PACKAGE_REPACKED_CANONICAL_MEMBERS"
else:
    raise RuntimeError(f"Ambiguous Trial ZIPs: {trial_zips}")
RESOLVED["trial_zip"] = TRIAL_ZIP.resolve()
ASR_ROOT = RESOLVED["asr_provenance"].parent
E5_ROOT = RESOLVED["e5_model"].parent
XCLIP_ROOT = RESOLVED["xclip_config"].parent
QWEN_ROOT = RESOLVED["qwen_config"].parent
print(
    {**{key: str(value) for key, value in RESOLVED.items()}, "trial_source_mode": TRIAL_SOURCE_MODE}
)

In [ ]:
test_env = os.environ.copy()
test_env["PYTHONPATH"] = str(REPO_DIR / "src") + (
    os.pathsep + test_env["PYTHONPATH"] if test_env.get("PYTHONPATH") else ""
)
test = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/unit/trial_p1/test_multimodal_dryrun.py",
        "tests/unit/trial_p1/test_r2_policy.py",
        "tests/unit/trial_p1/test_r3_policy.py",
        "tests/unit/trial_p1/test_r4_policy.py",
        "tests/unit/fs1_v11",
        "tests/unit/external_multimodal_v3",
        "-q",
    ],
    cwd=REPO_DIR,
    env=test_env,
    capture_output=True,
    text=True,
)
TEST_SUMMARY = {
    "returncode": test.returncode,
    "stdout_tail": test.stdout.splitlines()[-20:],
    "stderr_tail": test.stderr.splitlines()[-20:],
}
if test.returncode:
    raise RuntimeError(TEST_SUMMARY)
print(TEST_SUMMARY)

In [ ]:
from aic2026_eval.census import build_corpus_inventory
from triage_eg.trial_p1 import compile_queries, parse_trial_zip
from triage_eg.trial_p1.multimodal_dryrun import (
    CanonicalBTCFrameMapper,
    build_asr_candidate_evidence,
    build_external_parquet_evidence,
    normalize_trial_plans,
    read_jsonl,
    run_causal_graph_fixture,
    validate_trial_runtime_assets,
)
from triage_eg.trial_p1.r3_policy import (
    build_bounded_qa_rescue_evidence,
    derive_r3_anchor_profiles,
)

PLANS = read_jsonl(RESOLVED["query_plans"])
QUERIES = normalize_trial_plans(PLANS)
OFFICIAL_QUERIES = normalize_trial_plans(compile_queries(parse_trial_zip(RESOLVED["trial_zip"])))
BCF1 = read_jsonl(RESOLVED["bcf1_predictions"])
if QUERIES != OFFICIAL_QUERIES:
    raise RuntimeError("TRIAL_FROZEN_PLANS_DO_NOT_MATCH_OFFICIAL_PACKAGE")
INVENTORY, INVENTORY_SUMMARY, INVENTORY_ISSUES = build_corpus_inventory(RAW_INPUT)
if INVENTORY_SUMMARY["status"] != "PASS" or len(INVENTORY) != 873:
    raise RuntimeError(
        {"TRIAL_CANONICAL_INVENTORY_FAILED": INVENTORY_SUMMARY, "issues": INVENTORY_ISSUES[:20]}
    )
BTC_MAPPER = CanonicalBTCFrameMapper(INVENTORY)
ASSET_VALIDATION = validate_trial_runtime_assets(
    bcf1_predictions=RESOLVED["bcf1_predictions"],
    xclip_root=XCLIP_ROOT,
    e5_root=E5_ROOT,
    qwen_root=QWEN_ROOT,
)
CAUSAL_FIXTURE = run_causal_graph_fixture()
if CAUSAL_FIXTURE["status"] != "PASS":
    raise RuntimeError({"TRIAL_CAUSAL_FIXTURE_FAILED": CAUSAL_FIXTURE})
from triage_eg.trial_p1.asr_v12_loader import (  # noqa: E402
    ASR_EXTERNAL_V3_SOURCE_TYPE,
    load_asr_evidence,
)

ASR_LOADER = load_asr_evidence(ASR_ROOT, ASR_EXTERNAL_V3_SOURCE_TYPE)
from triage_eg.external_multimodal_v3.trial_smoke import (  # noqa: E402
    OnnxE5QueryEncoder,
    _e5_search,
)

encoder = OnnxE5QueryEncoder(E5_ROOT, exact_revision="03415a4be176a1620747c692ed433219fabc3def")
query_texts = [str(row.get("query", "")) for row in QUERIES]
e5_lists = _e5_search(ASR_LOADER, query_texts, encoder, 200)
E5 = {row["query_id"]: hits for row, hits in zip(QUERIES, e5_lists, strict=True)}
EVIDENCE = {
    "asr": build_asr_candidate_evidence(
        QUERIES, ASR_LOADER, canonical_mapper=BTC_MAPPER, e5_results=E5
    ),
    "ocr": build_external_parquet_evidence(QUERIES, RESOLVED["ocr_corpus"], "ocr"),
    "object": build_external_parquet_evidence(QUERIES, RESOLVED["object_corpus"], "object"),
    "action": {},
    "action_revision": {},
    "qwen": {},
}
R3_ANCHOR_PROFILES = derive_r3_anchor_profiles(QUERIES, PLANS)
QA_RESCUE, QA_RESCUE_DIAGNOSTICS = build_bounded_qa_rescue_evidence(
    QUERIES,
    R3_ANCHOR_PROFILES,
    ASR_LOADER,
    RESOLVED["ocr_corpus"],
    BTC_MAPPER,
    EVIDENCE,
    BCF1,
)
R3_EVIDENCE = {
    name: {query_id: [dict(row) for row in rows] for query_id, rows in values.items()}
    for name, values in EVIDENCE.items()
}
for modality in ("asr", "ocr"):
    for query_id, rescue_rows in QA_RESCUE[modality].items():
        merged = [*rescue_rows, *R3_EVIDENCE[modality].get(query_id, [])]
        selected, seen = [], set()
        for row in merged:
            identity = (
                str(row["video_id"]),
                int(row["frame_id"]),
                str(row.get("text") or (row.get("asr_span") or {}).get("text", "")),
            )
            if identity in seen:
                continue
            seen.add(identity)
            selected.append({**row, "rank": len(selected) + 1})
        R3_EVIDENCE[modality][query_id] = selected
print(
    {
        "queries": len(QUERIES),
        "official_trial_match": True,
        "trial_source_mode": TRIAL_SOURCE_MODE,
        "bcf1_rows": len(BCF1),
        "inventory": INVENTORY_SUMMARY,
        "asset_validation": ASSET_VALIDATION["status"],
        "causal_fixture": CAUSAL_FIXTURE["status"],
        "asr_rows_r2": sum(map(len, EVIDENCE["asr"].values())),
        "asr_rows_r3": sum(map(len, R3_EVIDENCE["asr"].values())),
        "ocr_rows_r2": sum(map(len, EVIDENCE["ocr"].values())),
        "ocr_rows_r3": sum(map(len, R3_EVIDENCE["ocr"].values())),
        "object_rows": sum(map(len, EVIDENCE["object"].values())),
        "qa_rescue": QA_RESCUE_DIAGNOSTICS,
        "e5_encoder_provenance": encoder.provenance,
        "corpus_job_run": False,
    }
)


In [ ]:
from triage_eg.data.stage0_audit.asset_resolver import discover_layout, resolve_assets
from triage_eg.fs1_v11.xclip import XClipAdapter, uniform_indices
from triage_eg.trial_p1.multimodal_dryrun import (
    build_xclip_event_evidence,
    build_xclip_revision_evidence,
)
from triage_eg.video import OpenCVRawVideoDecoder

video_parts, keyframe_parts = discover_layout(RAW_INPUT)
xclip = XClipAdapter(XCLIP_ROOT)
xclip.load()
XCLIP_FAILURES = []


def score_window(text, video_id, center):
    decoder = None
    try:
        assets = resolve_assets(RAW_INPUT, video_id, video_parts, keyframe_parts)
        decoder = OpenCVRawVideoDecoder(video_id, assets.video)
        actual_center = max(0, min(int(center), decoder.info.total_frames - 1))
        start = max(0, actual_center - 48)
        end = min(decoder.info.total_frames - 1, actual_center + 48)
        indices = uniform_indices(start, end)
        decoded = decoder.decode_indices(indices)
        if len(decoded) != 8:
            raise RuntimeError(f"XCLIP_DECODED_FRAME_COUNT_INVALID:{len(decoded)}")
        frames = [row.image for row in decoded]
    except (IndexError, OSError, RuntimeError) as error:
        XCLIP_FAILURES.append(
            {
                "stage": "xclip_decode",
                "video_id": video_id,
                "requested_center": int(center),
                "error": f"{type(error).__name__}: {error}",
            }
        )
        return {"finite": False, "decode_failed": True}
    finally:
        if decoder is not None:
            decoder.close()
    return {**xclip.score(text, frames), "center_frame_id": actual_center}


EVIDENCE["action"] = build_xclip_event_evidence(QUERIES, BCF1, score_window, candidates_per_event=8)
EVIDENCE["action_revision"] = build_xclip_revision_evidence(
    QUERIES, BCF1, EVIDENCE["action"], score_window, candidates_per_event=8
)
R3_EVIDENCE["action"] = EVIDENCE["action"]
R3_EVIDENCE["action_revision"] = EVIDENCE["action_revision"]
xclip.unload()

# R4 local OCR is query-local and bounded to shortlisted candidate/neighbor frames.
from PIL import Image
from triage_eg.trial_p1.r4_policy import build_bounded_local_ocr_rescue

TESSERACT = shutil.which("tesseract")
TESSERACT_LANG = "eng"
TESSERACT_VERSION = None
if TESSERACT:
    version_lines = subprocess.run(
        [TESSERACT, "--version"], capture_output=True, text=True, check=False
    ).stdout.splitlines()
    TESSERACT_VERSION = version_lines[0] if version_lines else "UNKNOWN"
    lang_probe = subprocess.run(
        [TESSERACT, "--list-langs"], capture_output=True, text=True, check=False
    ).stdout.splitlines()
    if "vie" in lang_probe:
        TESSERACT_LANG = "vie+eng" if "eng" in lang_probe else "vie"
LOCAL_OCR_IMAGE_ROOT = WORK_ROOT / "local_ocr_frames"
LOCAL_OCR_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)


def local_ocr_provider(video_id, frame_ids):
    if not TESSERACT:
        raise RuntimeError("R4_LOCAL_TESSERACT_NOT_AVAILABLE")
    assets = resolve_assets(RAW_INPUT, video_id, video_parts, keyframe_parts)
    decoder = OpenCVRawVideoDecoder(video_id, assets.video)
    try:
        valid = sorted(
            {max(0, min(int(value), decoder.info.total_frames - 1)) for value in frame_ids}
        )
        decoded = decoder.decode_indices(valid)
    finally:
        decoder.close()
    rows = []
    for item in decoded:
        image_path = LOCAL_OCR_IMAGE_ROOT / f"{video_id}_{item.actual_frame_idx}.png"
        Image.fromarray(item.image).save(image_path)
        try:
            process = subprocess.run(
                [TESSERACT, str(image_path), "stdout", "-l", TESSERACT_LANG, "--psm", "6"],
                capture_output=True, text=True, check=False, timeout=60,
            )
        except subprocess.TimeoutExpired as error:
            raise RuntimeError(f"TESSERACT_TIMEOUT:{item.actual_frame_idx}") from error
        finally:
            image_path.unlink(missing_ok=True)
        if process.returncode:
            raise RuntimeError(f"TESSERACT_FAILED:{process.stderr[-500:]}")
        text = " | ".join(line.strip() for line in process.stdout.splitlines() if line.strip())
        rows.append(
            {"frame_id": item.actual_frame_idx, "text": text, "confidence": None,
             "engine": f"tesseract:{TESSERACT_LANG}"}
        )
    return rows


LOCAL_OCR_ROWS, LOCAL_OCR_AUDIT = build_bounded_local_ocr_rescue(
    QUERIES, R3_ANCHOR_PROFILES, R3_EVIDENCE, QA_RESCUE_DIAGNOSTICS, local_ocr_provider
)
for query_id, rows in LOCAL_OCR_ROWS.items():
    merged = [*rows, *R3_EVIDENCE["ocr"].get(query_id, [])]
    selected, seen = [], set()
    for row in merged:
        identity = (str(row["video_id"]), int(row["frame_id"]), str(row.get("text", "")))
        if identity in seen:
            continue
        seen.add(identity)
        selected.append({**row, "rank": len(selected) + 1})
    R3_EVIDENCE["ocr"][query_id] = selected
LOCAL_OCR_FAILURES = [
    {"stage": "local_ocr_r4", "query_id": row["query_id"], "error": error}
    for row in LOCAL_OCR_AUDIT for error in row["failures"]
]
print(
    {
        "xclip_event_rows": sum(map(len, EVIDENCE["action"].values())),
        "revision_rows": sum(map(len, EVIDENCE["action_revision"].values())),
        "revision_modes": {
            key: [row["revision_search_mode"] for row in value]
            for key, value in EVIDENCE["action_revision"].items()
        },
        "per_trake": {key: len(value) for key, value in EVIDENCE["action"].items()},
        "candidate_decode_failures": len(XCLIP_FAILURES),
        "local_ocr_engine": TESSERACT,
        "local_ocr_language": TESSERACT_LANG,
        "local_ocr_version": TESSERACT_VERSION,
        "local_ocr_rows": sum(map(len, LOCAL_OCR_ROWS.values())),
        "local_ocr_audit": LOCAL_OCR_AUDIT,
    }
)

In [ ]:
from PIL import Image

from triage_eg.fs1.qa import GroundingCandidate, bounded_grounding_candidates
from triage_eg.fs1.qwen_adapter import QwenEvidenceAdapter
from triage_eg.fs1_v11.pipeline import grouped
from triage_eg.trial_p1.multimodal_dryrun import build_qwen_context, select_qwen_grounding_rows
from triage_eg.trial_p1.r3_policy import augment_qa_context_r3
from triage_eg.trial_p1.r4_policy import verify_answer_r4

baseline = grouped(BCF1)
qwen = QwenEvidenceAdapter(QWEN_ROOT)
qwen.load()
QA_EXTRACTIONS = []
QA_VERIFICATIONS_R4 = []
QWEN_FAILURES = []


def run_qwen_pass(query, evidence, label):
    query_id = str(query["query_id"])
    grounding = select_qwen_grounding_rows(
        ocr_rows=evidence["ocr"][query_id],
        asr_rows=evidence["asr"][query_id],
        baseline_rows=baseline[query_id],
    )
    candidates = bounded_grounding_candidates(
        [
            GroundingCandidate(
                str(row["video_id"]),
                int(row["frame_id"]),
                int(row.get("rank", 100)),
                {
                    "grounding_source": row["grounding_source"],
                    "grounding_source_rank": row["grounding_source_rank"],
                },
            )
            for row in grounding
        ]
    )
    answers = []
    for candidate in candidates:
        candidate_seconds = BTC_MAPPER.frame_seconds(candidate.video_id, candidate.frame_id)
        context_text, context_rows = build_qwen_context(
            {"video_id": candidate.video_id, "frame_id": candidate.frame_id},
            ocr_rows=evidence["ocr"][query_id],
            asr_rows=evidence["asr"][query_id],
            candidate_seconds=candidate_seconds,
        )
        visual_source = {
            "source_id": f"visual:{candidate.video_id}:{candidate.frame_id}",
            "modality": "visual",
            "video_id": candidate.video_id,
            "frame_id": candidate.frame_id,
            "distance_frames": 0,
            "time_distance_seconds": 0.0,
            "text": "",
            "confidence": None,
        }
        evidence_catalog = [visual_source, *context_rows]
        if label == "R4":
            context_rows = augment_qa_context_r3(
                R3_ANCHOR_PROFILES[query_id],
                candidate.video_id,
                context_rows,
                {
                    "asr": evidence["asr"][query_id],
                    "ocr": evidence["ocr"][query_id],
                },
            )
            context_text = " | ".join(
                f"[{row['source_id']}|{row['modality']}] {row['text']}"
                for row in context_rows
            )
            evidence_catalog = [visual_source, *context_rows]
        decoder = None
        try:
            assets = resolve_assets(RAW_INPUT, candidate.video_id, video_parts, keyframe_parts)
            decoder = OpenCVRawVideoDecoder(candidate.video_id, assets.video)
            if not 0 <= candidate.frame_id < decoder.info.total_frames:
                raise IndexError(
                    f"QWEN_CANDIDATE_FRAME_OUT_OF_BOUNDS:{candidate.frame_id}:"
                    f"{decoder.info.total_frames}"
                )
            image = Image.fromarray(decoder.decode_indices([candidate.frame_id])[0].image)
        except (IndexError, OSError, RuntimeError) as error:
            failure = {
                "stage": "qwen_decode",
                "pass": label,
                "query_id": query_id,
                "video_id": candidate.video_id,
                "frame_id": candidate.frame_id,
                "error": f"{type(error).__name__}: {error}",
            }
            QWEN_FAILURES.append(failure)
            QA_EXTRACTIONS.append(failure)
            continue
        finally:
            if decoder is not None:
                decoder.close()
        kind = str(query.get("answer_type", "OTHER"))
        policy = str(query.get("answer_policy", "SHORT_SEMANTIC"))
        extraction, audit = qwen.answer_extraction(
            candidate,
            image,
            description=str(query["query"]),
            question=str(query["query"]),
            evidence_rows=evidence_catalog,
            answer_type=kind,
            answer_policy=policy,
        )
        QA_EXTRACTIONS.append({"query_id": query_id, "pass": label, "audit": audit})
        if extraction is None:
            failure = verify_answer_r4(
                None, kind, evidence_catalog, R3_ANCHOR_PROFILES[query_id],
                grounding_plausibility=1.0 / (1.0 + candidate.evidence_rank),
            )
            failure.update({"query_id": query_id, "video_id": candidate.video_id,
                            "frame_id": candidate.frame_id, "rank": candidate.evidence_rank})
            QA_VERIFICATIONS_R4.append(failure)
            continue
        plausibility = 1.0 / (1.0 + candidate.evidence_rank)
        verified = verify_answer_r4(
                extraction,
                kind,
                evidence_catalog,
                R3_ANCHOR_PROFILES[query_id],
                grounding_plausibility=plausibility,
        )
        verified.update(
            {
                "query_id": query_id,
                "rank": candidate.evidence_rank,
                "source": f"qwen_{label.casefold()}_verified_extraction",
                "evidence_sufficient": verified["final_evidence_sufficient"],
                "evidence_context": context_text,
                "evidence_rows": evidence_catalog,
                "evidence_sources": verified["corroborating_modalities"],
                "grounding_source": candidate.evidence["grounding_source"],
                "grounding_source_rank": candidate.evidence["grounding_source_rank"],
            }
        )
        QA_VERIFICATIONS_R4.append(dict(verified))
        answers.append(verified)
    return answers


R4_QWEN = {}
for query in (row for row in QUERIES if row["task"] == "QA"):
    query_id = str(query["query_id"])
    R4_QWEN[query_id] = run_qwen_pass(query, R3_EVIDENCE, "R4")
qwen.unload()
R3_EVIDENCE["qwen"] = R4_QWEN
print(
    {
        "r4_qwen_verified_sufficient": {
            key: sum(row["final_evidence_sufficient"] for row in value)
            for key, value in R4_QWEN.items()
        },
        "candidate_decode_failures": len(QWEN_FAILURES),
    }
)


In [ ]:
from triage_eg.trial_p1.multimodal_dryrun import (
    select_novel_graph_revision,
    sha256_file,
    write_jsonl,
)
from triage_eg.trial_p1.r3_policy import (
    tier_evidence_r3,
)
from triage_eg.trial_p1.r4_policy import build_r4_candidates, write_r4_artifacts

R3_TIERED_EVIDENCE, R3_TIER_DIAGNOSTICS = tier_evidence_r3(
    QUERIES,
    R3_EVIDENCE,
    R3_ANCHOR_PROFILES,
    baseline_rows=BCF1,
)


def r4_revision_provider(query, event, action):
    return select_novel_graph_revision(
        query,
        event.event_index,
        baseline_rows=baseline[query["query_id"]],
        action_rows=R3_TIERED_EVIDENCE["action"].get(query["query_id"], []),
        revision_rows=R3_TIERED_EVIDENCE["action_revision"].get(query["query_id"], []),
    )


RUNTIME_FAILURES = [*XCLIP_FAILURES, *LOCAL_OCR_FAILURES, *QWEN_FAILURES]
try:
    R4_RESULT = build_r4_candidates(
        QUERIES,
        BCF1,
        R3_TIERED_EVIDENCE,
        R4_QWEN,
        r4_revision_provider,
        inventory=INVENTORY,
    )
except RuntimeError as error:
    failure = {
        "error": f"{type(error).__name__}: {error}",
        "HEAD": HEAD,
        "whisper_run": False,
        "corpus_job_run": False,
        "gt_opened": False,
        "submission_uploaded": False,
    }
    (OUTPUT_ROOT / "r4_candidate_build_failure.json").write_text(
        json.dumps(failure, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )
    write_jsonl(OUTPUT_ROOT / "asr_r3_specificity_diagnostics.jsonl", R3_TIER_DIAGNOSTICS)
    write_jsonl(OUTPUT_ROOT / "qa_r4_extractions.jsonl", QA_EXTRACTIONS)
    write_jsonl(OUTPUT_ROOT / "qa_r4_context_relevance.jsonl", QA_VERIFICATIONS_R4)
    shutil.make_archive(str(OUTPUT_ZIP.with_suffix("")), "zip", OUTPUT_ROOT)
    print({**failure, "diagnostic_zip": str(OUTPUT_ZIP)})
    raise
ASSET_HASHES = {
    key: sha256_file(path)
    for key, path in RESOLVED.items()
    if Path(path).is_file()
}
write_jsonl(OUTPUT_ROOT / "runtime_candidate_failures.jsonl", RUNTIME_FAILURES)
(OUTPUT_ROOT / "asset_validation.json").write_text(
    json.dumps(ASSET_VALIDATION, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
(OUTPUT_ROOT / "causal_graph_fixture.json").write_text(
    json.dumps(CAUSAL_FIXTURE, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
(OUTPUT_ROOT / "inventory_summary.json").write_text(
    json.dumps(
        {"summary": INVENTORY_SUMMARY, "issues": INVENTORY_ISSUES},
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)
REPORT = write_r4_artifacts(
    OUTPUT_ROOT,
    QUERIES,
    BCF1,
    R4_RESULT,
    R3_ANCHOR_PROFILES,
    QA_EXTRACTIONS,
    QA_VERIFICATIONS_R4,
    LOCAL_OCR_AUDIT,
    provenance={
        "mode": "TRIAL_P1_R4_SURGICAL_FINAL_REPAIR",
        "HEAD": HEAD,
        "trial_source_mode": TRIAL_SOURCE_MODE,
        "official_trial_zip": str(RESOLVED["trial_zip"]),
        "official_trial_sha256": sha256_file(RESOLVED["trial_zip"]),
        "official_trial_matches_frozen_plans": True,
        "true_bcf1_sha256": sha256_file(RESOLVED["bcf1_predictions"]),
        "asset_hashes": ASSET_HASHES,
        "asset_validation": ASSET_VALIDATION,
        "canonical_inventory": INVENTORY_SUMMARY,
        "causal_fixture": CAUSAL_FIXTURE,
        "runtime_candidate_failure_count": len(RUNTIME_FAILURES),
        "qwen_parse_failure_count": sum(
            1
            for row in QA_EXTRACTIONS
            if isinstance(row.get("audit"), dict) and row["audit"].get("parse_reason")
        ),
        "asr_source_type": "ASR_EXTERNAL_V3_VALIDATED",
        "e5_query_encoder": encoder.provenance,
        "local_ocr": {"executable": TESSERACT, "language": TESSERACT_LANG,
                      "version": TESSERACT_VERSION, "bounded_audit": LOCAL_OCR_AUDIT},
        "whisper_run": False,
        "corpus_job_run": False,
        "gt_opened": False,
        "submission_uploaded": False,
        "production_policy_changed": False,
        "multimodal_runtime_changed": False,
        "event_graph_changed": False,
    },
)
shutil.make_archive(str(OUTPUT_ZIP.with_suffix("")), "zip", OUTPUT_ROOT)
print(
    {
        "recommendation": REPORT["recommendation"],
        "hard_automated_gates_pass": REPORT["hard_automated_gates_pass"],
        "hard_gates": REPORT["hard_gates"],
        "oj_ready_submissions": REPORT["oj_ready_submissions"],
        "output_zip": str(OUTPUT_ZIP),
        "output_zip_sha256": sha256_file(OUTPUT_ZIP),
        "whisper_run": False,
        "corpus_job_run": False,
        "gt_opened": False,
        "submission_uploaded": False,
        "cross_l21": "NOT_RUN",
    }
)
